In [ ]:
import requests
import pandas as pd
import time
import os
import re
from bs4 import BeautifulSoup


# ============================================================
# 0. 파일 번호 설정
# ============================================================
# 첫 번째 실행 → 1
# 두 번째 실행 → 2
# 세 번째 실행 → 3

FILE_NUMBER = 1


# ============================================================
# 1. Steam API 설정
# ============================================================

API_KEY = "B1FC36C7D790B5F17DCD8E33F5C33DF2"


# ============================================================
# 2. 수집할 SteamID
# ============================================================

STEAM_IDS = [
    "76561198056237344",
    "76561198089635514",
    "76561198993558694",
    "76561198164041869",
    "76561198325380788",
    "76561198409974659",
    "76561199033536730",
    "76561198421742763",
    "76561198282073896",
    "76561198403079074",
    "76561198084007540",
    "76561198267704889",
    "76561198373401513",
    "76561198327146957",
    "76561198170938615",
    "76561198406808917",
    "76561198112922943",
    "76561198335134165",
    "76561198975773802",
    "76561199070226625",
    "76561198139147257",
    "76561199226650132",
    "76561199439546632",
    "76561199109484126",
    "76561198120033470",
    "76561199151079821",
    "76561198255922415",
    "76561198168331079",
    "76561198097783552",
    "76561199812324187"
]


# ============================================================
# 3. 상위 게임 개수
# ============================================================

TOP_GAME_COUNT = 30


# ============================================================
# 4. API 주소
# ============================================================

OWNED_GAMES_URL = (
    "https://api.steampowered.com/"
    "IPlayerService/GetOwnedGames/v1/"
)

RECENT_GAMES_URL = (
    "https://api.steampowered.com/"
    "IPlayerService/GetRecentlyPlayedGames/v1/"
)

APP_DETAILS_URL = (
    "https://store.steampowered.com/api/appdetails"
)

STORE_PAGE_URL = (
    "https://store.steampowered.com/app/"
)


# ============================================================
# 5. Requests Session
# ============================================================

session = requests.Session()

session.headers.update({
    "User-Agent":
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/151.0.0.0 Safari/537.36"
})


# ============================================================
# 6. 게임 정보 캐시
# ============================================================

game_cache = {}


# ============================================================
# 7. 게임 장르 / 카테고리 가져오기
# ============================================================

def get_game_details(appid):

    # --------------------------------------------------------
    # 이미 가져온 게임이면 캐시 사용
    # --------------------------------------------------------

    if appid in game_cache:

        print(
            "    → 기존 장르/카테고리 사용"
        )

        return (
            game_cache[appid]["genre"],
            game_cache[appid]["categories"]
        )


    print(
        "    → Steam에서 장르/카테고리 가져오는 중..."
    )


    params = {
        "appids": appid,
        "l": "english"
    }


    try:

        response = session.get(
            APP_DETAILS_URL,
            params=params,
            timeout=15
        )


        if response.status_code != 200:

            return [], []


        data = response.json()


        app_data = data.get(
            str(appid),
            {}
        )


        if not app_data.get("success"):

            return [], []


        game_info = app_data.get(
            "data",
            {}
        )


        # ----------------------------------------------------
        # 장르
        # ----------------------------------------------------

        genres = game_info.get(
            "genres",
            []
        )


        genre_list = [

            genre.get("description")

            for genre in genres

            if genre.get("description")
        ]


        # ----------------------------------------------------
        # 카테고리
        # ----------------------------------------------------

        categories = game_info.get(
            "categories",
            []
        )


        category_list = [

            category.get("description")

            for category in categories

            if category.get("description")
        ]


        # ----------------------------------------------------
        # 캐시에 저장
        # ----------------------------------------------------

        game_cache[appid] = {

            "genre":
                genre_list,

            "categories":
                category_list,

            "tags":
                None
        }


        return (
            genre_list,
            category_list
        )


    except Exception as e:

        print(
            f"    장르/카테고리 오류: {e}"
        )

        return [], []


# ============================================================
# 8. Steam 사용자 태그 가져오기
# ============================================================

def get_steam_tags(appid):

    # --------------------------------------------------------
    # 캐시에 태그가 있으면 사용
    # --------------------------------------------------------

    if (
        appid in game_cache
        and game_cache[appid].get("tags") is not None
    ):

        print(
            "    → 기존 태그 사용"
        )

        return game_cache[appid]["tags"]


    print(
        "    → Steam에서 태그 가져오는 중..."
    )


    url = f"{STORE_PAGE_URL}{appid}/"


    try:

        response = session.get(
            url,
            timeout=15
        )


        if response.status_code != 200:

            return []


        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )


        tags = []


        # ----------------------------------------------------
        # 일반적인 Steam 태그 구조
        # ----------------------------------------------------

        tag_elements = soup.select(
            ".app_tag"
        )


        for element in tag_elements:

            tag = element.get_text(
                strip=True
            )


            if not tag:

                continue


            if tag == "+":

                continue


            if tag not in tags:

                tags.append(tag)


        # ----------------------------------------------------
        # HTML 구조가 달라졌을 경우
        # ----------------------------------------------------

        if not tags:

            patterns = [

                r'class="app_tag"[^>]*>\s*([^<]+)',

                r'class="app_tag[^"]*"[^>]*>\s*([^<]+)',

                r'data-tagid="[^"]+"[^>]*>\s*([^<]+)'
            ]


            html = response.text


            for pattern in patterns:

                matches = re.findall(
                    pattern,
                    html,
                    flags=re.IGNORECASE
                )


                for match in matches:

                    tag = (
                        match
                        .replace("&amp;", "&")
                        .strip()
                    )


                    if (
                        tag
                        and tag != "+"
                        and tag not in tags
                    ):

                        tags.append(tag)


        # ----------------------------------------------------
        # 태그 정리
        # ----------------------------------------------------

        cleaned_tags = []


        for tag in tags:

            tag = (
                tag
                .replace("\n", "")
                .replace("\r", "")
                .strip()
            )


            if (
                tag
                and tag != "+"
                and tag not in cleaned_tags
            ):

                cleaned_tags.append(tag)


        # ----------------------------------------------------
        # 캐시에 저장
        # ----------------------------------------------------

        if appid not in game_cache:

            game_cache[appid] = {

                "genre":
                    [],

                "categories":
                    [],

                "tags":
                    cleaned_tags
            }

        else:

            game_cache[appid]["tags"] = (
                cleaned_tags
            )


        return cleaned_tags


    except Exception as e:

        print(
            f"    태그 오류: {e}"
        )

        return []


# ============================================================
# 9. 전체 게임 데이터 수집
# ============================================================

all_games = []


print(
    "=" * 70
)

print(
    "Steam 플레이타임 상위 "
    f"{TOP_GAME_COUNT}개 게임 데이터 수집 시작"
)

print(
    "=" * 70
)


for user_index, steam_id in enumerate(
    STEAM_IDS,
    start=1
):


    print()

    print(
        f"[사용자 {user_index}/{len(STEAM_IDS)}]"
    )

    print(
        f"SteamID: {steam_id}"
    )


    # ========================================================
    # Steam 보유 게임 API
    # ========================================================

    params = {

        "key":
            API_KEY,

        "steamid":
            steam_id,

        "include_appinfo":
            True,

        "include_played_free_games":
            True,

        "format":
            "json"
    }


    try:

        response = session.get(
            OWNED_GAMES_URL,
            params=params,
            timeout=15
        )


        print(
            "상태 코드:",
            response.status_code
        )


        response.raise_for_status()


        data = response.json()


        games = data.get(
            "response",
            {}
        ).get(
            "games",
            []
        )


    except Exception as e:

        print(
            "보유 게임 데이터 오류:",
            e
        )

        continue


    # ========================================================
    # 게임이 없는 경우
    # ========================================================

    if not games:

        print(
            "→ 게임 데이터가 없습니다."
        )

        continue


    print(
        f"→ 전체 보유 게임: "
        f"{len(games)}개"
    )


    # ========================================================
    # 플레이타임 기준 내림차순 정렬
    # ========================================================

    games = sorted(

        games,

        key=lambda x: x.get(
            "playtime_forever",
            0
        ),

        reverse=True
    )


    # ========================================================
    # 상위 30개 선택
    #
    # 30개 초과:
    #     → 플레이타임 상위 30개
    #
    # 30개 이하:
    #     → 전체 게임
    # ========================================================

    top_games = games[
        :TOP_GAME_COUNT
    ]


    print(
        f"→ 실제 수집 게임: "
        f"{len(top_games)}개"
    )


    # ========================================================
    # 선택된 게임 데이터 수집
    # ========================================================

    for game_index, game in enumerate(
        top_games,
        start=1
    ):


        appid = game.get(
            "appid"
        )


        game_name = game.get(
            "name",
            "Unknown"
        )


        print()

        print(
            f"[{game_index}/{len(top_games)}] "
            f"{game_name}"
        )


        # ====================================================
        # 총 플레이타임 → 시간
        # ====================================================

        playtime_minutes = game.get(
            "playtime_forever",
            0
        )


        playtime_hours = round(

            playtime_minutes / 60,

            2
        )


        print(
            f"    플레이타임: "
            f"{playtime_hours}시간"
        )


        # ====================================================
        # 장르 / 카테고리
        # ====================================================

        genres, categories = (
            get_game_details(
                appid
            )
        )


        # ====================================================
        # 태그
        # ====================================================

        tags = get_steam_tags(
            appid
        )


        # ====================================================
        # 데이터 저장
        # ====================================================

        all_games.append({

            "steamid":
                steam_id,

            "appid":
                appid,

            "game_name":
                game_name,

            "playtime_hours":
                playtime_hours,

            "genre":
                ", ".join(genres)
                if genres
                else "Unknown",

            "tags":
                ", ".join(tags)
                if tags
                else "Unknown",

            "steam_categories":
                ", ".join(categories)
                if categories
                else "Unknown"
        })


        # ====================================================
        # API 요청 간격
        # ====================================================

        time.sleep(0.2)


# ============================================================
# 10. DataFrame 생성
# ============================================================

df_games = pd.DataFrame(
    all_games
)


# ============================================================
# 11. 최근 2주 플레이 데이터 수집
# ============================================================

recent_games = []


print()

print(
    "=" * 70
)

print(
    "최근 2주 플레이 데이터 수집"
)

print(
    "=" * 70
)


for user_index, steam_id in enumerate(
    STEAM_IDS,
    start=1
):


    print(
        f"\n[{user_index}/{len(STEAM_IDS)}]"
        f" {steam_id}"
    )


    params = {

        "key":
            API_KEY,

        "steamid":
            steam_id,

        "format":
            "json"
    }


    try:

        response = session.get(
            RECENT_GAMES_URL,
            params=params,
            timeout=15
        )


        print(
            "상태 코드:",
            response.status_code
        )


        response.raise_for_status()


        data = response.json()


        games = data.get(
            "response",
            {}
        ).get(
            "games",
            []
        )


    except Exception as e:

        print(
            "최근 플레이 데이터 오류:",
            e
        )

        continue


    # ========================================================
    # 최근 플레이 게임이 없는 경우
    # ========================================================

    if not games:

        print(
            "→ 최근 2주 플레이 게임 없음"
        )

        continue


    print(
        f"→ 최근 2주 플레이 게임: "
        f"{len(games)}개"
    )


    # ========================================================
    # 최근 플레이 데이터 저장
    # ========================================================

    for game in games:


        appid = game.get(
            "appid"
        )


        game_name = game.get(
            "name",
            "Unknown"
        )


        playtime_2weeks_minutes = game.get(
            "playtime_2weeks",
            0
        )


        playtime_2weeks_hours = round(

            playtime_2weeks_minutes / 60,

            2
        )


        recent_games.append({

            "steamid":
                steam_id,

            "appid":
                appid,

            "game_name":
                game_name,

            "recent_playtime_hours":
                playtime_2weeks_hours
        })


# ============================================================
# 12. 최근 플레이 DataFrame
# ============================================================

df_recent = pd.DataFrame(
    recent_games
)


# ============================================================
# 13. 전체 + 최근 플레이 데이터 결합
# ============================================================

if not df_games.empty:


    if not df_recent.empty:

        df_final = pd.merge(

            df_games,

            df_recent[
                [
                    "steamid",
                    "appid",
                    "recent_playtime_hours"
                ]
            ],

            on=[
                "steamid",
                "appid"
            ],

            how="left"
        )


    else:

        df_final = df_games.copy()


        df_final[
            "recent_playtime_hours"
        ] = 0


else:

    df_final = pd.DataFrame()


# ============================================================
# 14. 최근 플레이 기록이 없는 게임 → 0시간
# ============================================================

if not df_final.empty:

    df_final[
        "recent_playtime_hours"
    ] = df_final[
        "recent_playtime_hours"
    ].fillna(0)


# ============================================================
# 15. 컬럼 순서
# ============================================================

if not df_final.empty:

    df_final = df_final[
        [
            "steamid",
            "appid",
            "game_name",
            "playtime_hours",
            "recent_playtime_hours",
            "genre",
            "tags",
            "steam_categories"
        ]
    ]


# ============================================================
# 16. CSV 파일명
# ============================================================

GAME_FILE = (
    f"steam_user_games_{FILE_NUMBER}.csv"
)


RECENT_FILE = (
    f"steam_recent_games_{FILE_NUMBER}.csv"
)


# ============================================================
# 17. CSV 저장
# ============================================================

df_final.to_csv(

    GAME_FILE,

    index=False,

    encoding="utf-8-sig"
)


df_recent.to_csv(

    RECENT_FILE,

    index=False,

    encoding="utf-8-sig"
)


# ============================================================
# 18. 결과 확인
# ============================================================

print()

print(
    "=" * 70
)

print(
    "수집 완료"
)

print(
    "=" * 70
)


if not df_final.empty:


    print(
        "사용자 수:",
        df_final["steamid"].nunique()
    )


    print(
        "전체 저장 게임 데이터:",
        len(df_final)
    )


    print(
        "계정당 최대 게임 수:",
        TOP_GAME_COUNT
    )


    display(
        df_final.head(20)
    )


    print()

    print(
        "전체 게임 CSV:"
    )


    print(
        os.path.abspath(
            GAME_FILE
        )
    )


    # ========================================================
    # 태그 수집 성공률
    # ========================================================

    unknown_tags = (

        df_final["tags"]
        == "Unknown"

    ).sum()


    total_games = len(
        df_final
    )


    if total_games > 0:

        tag_success_rate = (

            (total_games - unknown_tags)

            / total_games

            * 100
        )


        print()

        print(
            f"태그 수집 성공률: "
            f"{tag_success_rate:.1f}%"
        )


# ============================================================
# 19. 최근 2주 플레이 데이터 결과
# ============================================================

print()

print(
    "=" * 70
)

print(
    "최근 2주 플레이 데이터"
)

print(
    "=" * 70
)


if not df_recent.empty:


    print(
        "최근 플레이 게임:",
        len(df_recent)
    )


    display(
        df_recent.head(20)
    )


    print()

    print(
        "최근 플레이 CSV:"
    )


    print(
        os.path.abspath(
            RECENT_FILE
        )
    )


else:

    print(
        "최근 2주 플레이 데이터가 없습니다."
    )


# ============================================================
# 20. 계정별 저장 게임 수 확인
# ============================================================

print()

print(
    "=" * 70
)

print(
    "계정별 저장 게임 수"
)

print(
    "=" * 70
)


if not df_final.empty:


    user_game_counts = (

        df_final

        .groupby(
            "steamid"
        )

        .size()

        .reset_index(
            name="game_count"
        )
    )


    print(
        user_game_counts.to_string(
            index=False
        )
    )

Steam 플레이타임 상위 30개 게임 데이터 수집 시작

[사용자 1/30]
SteamID: 76561198056237344
상태 코드: 200
→ 전체 보유 게임: 306개
→ 실제 수집 게임: 30개

[1/30] Team Fortress 2
    플레이타임: 650.33시간
    → Steam에서 장르/카테고리 가져오는 중...
    → Steam에서 태그 가져오는 중...

[2/30] Rocksmith® 2014 Edition - Remastered
    플레이타임: 486.5시간
    → Steam에서 장르/카테고리 가져오는 중...
    → Steam에서 태그 가져오는 중...

[3/30] Warframe
    플레이타임: 392.73시간
    → Steam에서 장르/카테고리 가져오는 중...
    → Steam에서 태그 가져오는 중...

[4/30] War Thunder
    플레이타임: 341.1시간
    → Steam에서 장르/카테고리 가져오는 중...
    → Steam에서 태그 가져오는 중...

[5/30] Counter-Strike 2
    플레이타임: 306.0시간
    → Steam에서 장르/카테고리 가져오는 중...
    → Steam에서 태그 가져오는 중...
